# Sapsucker Woods: reproducible acquisition and certification

**Kenneth Mwangi Kabiru · University of Leeds**

This notebook makes the scientific implementation visible. The cells below define window construction, Perch embeddings, LR/SVM fitting, entropy acquisition, stopping rules and paired analysis. Helpers hold file handling, exact audit mathematics and independent verification.

There are eight workflows, 20 histories per selected target, a 3,000-label training horizon, and one final audit per workflow. Unavailable and inadequate candidates remain in the analysis. The complete specification is `study.json`.

Upload the whole project folder to Drive. For fresh embeddings, select a GPU runtime and keep it throughout; the remaining work uses its CPU. A complete embedding cache can be reused from either CPU or GPU. No reference external results are bundled yet.

In [ ]:
# @title Configuration {vertical-output: true}
PROJECT_DIRECTORY = "/content/drive/MyDrive/ssw" # @param {type: 'string'}
DRIVE_CACHE_PATH = "/content/drive/MyDrive/ssw_cache" # @param {type: 'string'}
EMBEDDING_DEVICE = "gpu" # @param ["gpu", "cpu"]
REGENERATE_EMBEDDINGS = False # @param {type: 'boolean'}
ORIGINAL_ARCHIVE_PATH = "" # @param {type: 'string'}
AUDIO_BACKUP_DIRECTORY = "" # @param {type: 'string'}


## 1. Locate the project and prepare a separate run folder

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys
if any(p.startswith('/content/drive/') for p in [PROJECT_DIRECTORY,DRIVE_CACHE_PATH,AUDIO_BACKUP_DIRECTORY,ORIGINAL_ARCHIVE_PATH]):
    from google.colab import drive
    drive.mount('/content/drive')
PROJECT = Path(PROJECT_DIRECTORY).expanduser().resolve()
if not (PROJECT/'helpers').is_dir():
    raise FileNotFoundError('PROJECT_DIRECTORY must name the uploaded folder containing this notebook and helpers/')
spec = json.loads((PROJECT/'project.json').read_text())
package = json.loads((PROJECT/'manifest.json').read_text())
for name, expected in package['files'].items():
    if name.endswith('.ipynb'): continue  # Colab may save form values and cell outputs.
    if hashlib.sha256((PROJECT/name).read_bytes()).hexdigest() != expected:
        raise RuntimeError('Project file changed: '+name)
RUN_ROOT = (Path('/content/ssw_runs') if Path('/content').exists() else PROJECT.parent/'ssw_runs')/spec['study_sha256'][:16]/('regenerated' if REGENERATE_EMBEDDINGS else 'default')
RUN_ROOT = RUN_ROOT.resolve(); RUN_ROOT.mkdir(parents=True, exist_ok=True)
for p in (PROJECT/'helpers').glob('*.py'):
    destination=RUN_ROOT/p.name
    if destination.exists() and destination.read_bytes()!=p.read_bytes(): raise RuntimeError('Run directory belongs to different code')
    shutil.copyfile(p,destination)
for name in ['study.json','models.json','dataset.json','dependencies.json','project.json']:
    shutil.copyfile(PROJECT/name,RUN_ROOT/name)
for folder in ['data/source','data/audio','data/constructed','data/embeddings','results','runs','diagnostics']:
    (RUN_ROOT/folder).mkdir(parents=True,exist_ok=True)
os.chdir(RUN_ROOT)
print('Project located; visible implementation cells follow.')


The `%%writefile` cells contain the actual scientific code. They save it for execution in the pinned interpreters; they do not run the experiment until the execution cells below.

## 2. Construct windows and select targets
Five-second half-open windows use exact positive-duration annotation overlap. Unknown-call windows and windows containing zero-duration timestamps share one exclusion mask across all species. Target eligibility and all seeds are fixed before fitting.

In [ ]:
%%writefile construction.py
"""Construct a common window pool with unknown-call and ambiguous-timestamp exclusions."""
from artifacts import ROOT, check_recipe, dump, load, sha, savez, canonical_hash, utc
from collections import Counter
from decimal import Decimal, InvalidOperation
from pathlib import Path
import argparse, csv, json, re, traceback

def intersected_windows(start, end, count):
    """Exact decimal positive-duration overlap with half-open 5 s windows."""
    a, b = Decimal(str(start)), Decimal(str(end))
    if not a.is_finite() or not b.is_finite() or a < 0 or b <= a:
        raise ValueError('Malformed annotation interval: ' + repr((start, end)))
    # The first/last possible indices are checked by the defining inequality.
    first, last = max(0, int(a // 5)), min(count-1, int(b // 5))
    return [j for j in range(first, last+1) if max(Decimal(5*j), a) < min(Decimal(5*(j+1)), b)]

def zero_duration_window(timestamp, count):
    """The unique half-open complete window containing t, or None in the tail/end."""
    t = Decimal(str(timestamp))
    if not t.is_finite() or t < 0:raise ValueError('Invalid annotation timestamp')
    j = int(t // 5)
    return j if j < count else None


def preflight(root=ROOT):
    """Validate all source rows before audio transfer or model loading; no NumPy needed."""
    import hashlib
    recipe = check_recipe(root)
    for name in ['annotations.csv','species.csv','recording_location.txt']:
        if hashlib.md5((root/'data/source'/name).read_bytes()).hexdigest() != recipe['dataset']['files'][name]:
            raise RuntimeError('Pinned source checksum mismatch: '+name)
    inventory = {r['filename'] for r in load(root/'data/constructed/RECORDING_PERMUTATION.json')['inventory']}
    durations = {}
    selection = root/'data/constructed/RECORDING_SELECTION.json'
    if selection.exists():
        for r in load(selection)['selected']:
            durations[r['filename']] = Decimal(r['audio']['frames']) / Decimal(r['audio']['sample_rate'])
    with (root/'data/source/species.csv').open(newline='',encoding='utf-8-sig') as f:
        reader=csv.DictReader(f)
        if reader.fieldnames != ['Species eBird Code','Scientific Name','Common Name']:raise ValueError('Unexpected species schema')
        species={}
        for row in reader:
            code=row['Species eBird Code']
            if None in row or not code or code in species or not row['Scientific Name'] or not row['Common Name']:
                raise ValueError('Invalid species identifier or names')
            species[code]=row
    expected=['Filename','Start Time (s)','End Time (s)','Low Freq (Hz)','High Freq (Hz)','Species eBird Code']
    zeros=[];rows=0
    with (root/'data/source/annotations.csv').open(newline='',encoding='utf-8-sig') as f:
        reader=csv.DictReader(f)
        if reader.fieldnames!=expected:raise ValueError('Unexpected annotation schema')
        for line,row in enumerate(reader,2):
            rows+=1
            try:
                if None in row or any(row[k] is None for k in expected):raise ValueError('Malformed annotation columns')
                name,code=row['Filename'],row['Species eBird Code']
                if name not in inventory or (code not in species and code!='????'):raise ValueError('Unknown recording or species identifier')
                a,b=Decimal(row['Start Time (s)']),Decimal(row['End Time (s)'])
                if not a.is_finite() or not b.is_finite() or a<0 or b<a:raise ValueError('Invalid annotation times')
                if name in durations and b>durations[name]:raise ValueError('Annotation ends after selected audio duration')
                if a==b:zeros.append({'csv_line':line,'row':dict(row)})
            except (ValueError,InvalidOperation,TypeError,KeyError) as error:
                raise ValueError('Annotation line %d: %s; row=%r' % (line,error,row)) from error
    result={'status':'PASS','annotation_rows':rows,'zero_duration_rows':zeros,
            'policy':'Exclude the containing complete five-second window across all target species; no invented duration',
            'source_sha256':{n:sha(root/'data/source'/n) for n in ['annotations.csv','species.csv','recording_location.txt']}}
    destination=root/'data/constructed/ANNOTATION_PREFLIGHT.json'
    if destination.exists() and load(destination)!=result:raise RuntimeError('Annotation preflight changed on resume')
    dump(destination,result)
    return result


def eligible(positives, recordings, flagged_count):
    return positives >= 100 and recordings >= 3 and 9*positives <= 10*flagged_count

def task_seed(target_index, replicate):
    if not 1 <= target_index <= 5 or not 1 <= replicate <= 20:
        raise ValueError('Task/replicate outside frozen scope')
    return 991210000 + 100*target_index + replicate

def construct(root=ROOT):
    import numpy as np
    recipe = check_recipe(root)
    table_check = preflight(root)
    if np.__version__ != '1.24.4':
        raise RuntimeError('Construction requires NumPy 1.24.4')
    selection = load(root/'data/constructed/RECORDING_SELECTION.json')
    if selection['status'] != 'RECORDINGS_COMPLETE' or len(selection['selected']) != 33:
        raise RuntimeError('33 fully validated recordings are required before annotation construction')
    # Recheck immutable extracted bytes; no redrawing can occur during labeling.
    records = sorted(selection['selected'], key=lambda x: x['filename'])
    for record in records:
        if sha(root/'data/audio'/record['filename']) != record['audio']['sha256']:
            raise RuntimeError('Selected audio bytes changed')
    with (root/'data/source/species.csv').open(newline='', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        if reader.fieldnames != ['Species eBird Code', 'Scientific Name', 'Common Name']:
            raise ValueError('Unexpected species schema')
        species = {}
        for row in reader:
            code = row['Species eBird Code']
            if not code or code in species or not row['Scientific Name'] or not row['Common Name']:
                raise ValueError('Ambiguous species authority')
            species[code] = {'code': code, 'scientific_name': row['Scientific Name'], 'common_name': row['Common Name']}
    if '????' in species:
        species.pop('????')
    codes = sorted(species); code_index = {c:i for i,c in enumerate(codes)}
    record_index = {r['filename']:i for i,r in enumerate(records)}
    offsets = np.cumsum([0]+[r['audio']['complete_windows'] for r in records])
    N_raw = int(offsets[-1]); labels = np.zeros((N_raw,len(codes)),dtype=np.int8)
    unknown = np.zeros(N_raw,dtype=bool); ambiguous = np.zeros(N_raw,dtype=bool); window_ids = []; window_record = []; window_j = []
    for ri, record in enumerate(records):
        for j in range(record['audio']['complete_windows']):
            window_ids.append(record['filename']+'__'+f'{j:03d}')
            window_record.append(ri); window_j.append(j)
    inventory = {r['filename'] for r in load(root/'data/constructed/RECORDING_PERMUTATION.json')['inventory']}
    duplicate_rows = 0; seen = set(); tail_only = Counter(); rows_selected = 0; rows_total = 0; zero_rows = []
    with (root/'data/source/annotations.csv').open(newline='', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        expected = ['Filename','Start Time (s)','End Time (s)','Low Freq (Hz)','High Freq (Hz)','Species eBird Code']
        if reader.fieldnames != expected:
            raise ValueError('Unexpected annotation schema')
        for line, row in enumerate(reader,2):
            try:
                rows_total += 1
                name, code = row['Filename'], row['Species eBird Code']
                if name not in inventory or (code not in species and code != '????'):
                    raise ValueError('Missing or unknown recording/species identifier')
                a,b = Decimal(row['Start Time (s)']),Decimal(row['End Time (s)'])
                if not a.is_finite() or not b.is_finite() or a<0 or b<a:
                    raise ValueError('Invalid annotation times')
                key = tuple(row[c] for c in expected)
                duplicate_rows += key in seen; seen.add(key)
                zero_entry = None
                if a == b:
                    zero_entry = {'csv_line': line, 'row': dict(row), 'selected_recording': name in record_index,
                                  'window_id': None, 'already_unknown_excluded': None}
                    zero_rows.append(zero_entry)
                if name not in record_index:
                    continue
                rows_selected += 1; ri = record_index[name]; record = records[ri]
                duration = Decimal(record['audio']['frames'])/Decimal(record['audio']['sample_rate'])
                if b > duration:
                    raise ValueError('Annotation ends after selected audio duration')
                if a == b:
                    j = zero_duration_window(a, record['audio']['complete_windows'])
                    if j is not None:
                        ambiguous[offsets[ri]+j] = True
                        zero_entry['window_id'] = window_ids[offsets[ri]+j]
                    continue  # No invented interval or positive label.
                js = intersected_windows(a,b,record['audio']['complete_windows'])
                if not js:
                    tail_only[code] += 1
                positions = np.asarray(js,dtype=np.int64)+offsets[ri]
                if code == '????':
                    unknown[positions] = True
                else:
                    labels[positions,code_index[code]] = 1
            except (ValueError, InvalidOperation, TypeError, KeyError) as error:
                raise ValueError('Annotation line %d: %s; row=%r' % (line,error,row)) from error
    common_exclusion = unknown | ambiguous
    id_index = {name:i for i,name in enumerate(window_ids)}
    for item in zero_rows:
        if item['window_id'] is not None:
            item['already_unknown_excluded'] = bool(unknown[id_index[item['window_id']]])
    keep = ~common_exclusion; N = int(keep.sum()); B=N//4; K=N//10
    if N<3000 or B<3000 or N<2000 or K<1:
        raise RuntimeError('CONSTRUCTION_FAILURE: common pool cannot support frozen horizon/monitoring contract')
    window_record = np.asarray(window_record,dtype=np.int64)
    eligibility = []
    for i,code in enumerate(codes):
        positives = int(labels[keep,i].sum())
        coverage = len(np.unique(window_record[keep & (labels[:,i]>0)]))
        eligibility.append(dict(species[code], positive_windows=positives, positive_recordings=coverage,
                                removed_positive_windows=int(labels[common_exclusion,i].sum()),
                                support_pass=positives>=100,coverage_pass=coverage>=3,
                                geometry_pass=9*positives<=10*K,eligible=eligible(positives,coverage,K)))
    eligible_codes = [item['code'] for item in eligibility if item['eligible']]
    permutation = np.random.Generator(np.random.PCG64(991200001)).permutation(len(eligible_codes)).tolist()
    targets = sorted([eligible_codes[i] for i in permutation[:5]])
    pool_ids = np.asarray(window_ids)[keep]
    if len(set(pool_ids.tolist())) != N:
        raise RuntimeError('Duplicate common-pool ID')
    task_table = [dict(next(i for i in eligibility if i['code']==code),target_index=t,
                       seeds=[task_seed(t,r) for r in range(1,21)]) for t,code in enumerate(targets,1)]
    result = {
        'status':'TASKS_CONSTRUCTED' if targets else 'CONSTRUCTION_FAILURE_NO_ELIGIBLE_TARGETS',
        'recipe_freeze_sha256':sha(root/'study.json'),'utc':utc(),
        'raw_complete_windows':N_raw,'unknown_windows_removed':int(unknown.sum()),
        'zero_duration_annotation_rows':len(zero_rows),
        'zero_duration_selected_annotation_rows':sum(r['selected_recording'] for r in zero_rows),
        'zero_duration_timestamp_windows':int(ambiguous.sum()),
        'zero_duration_windows_already_unknown':int((ambiguous & unknown).sum()),
        'zero_duration_additional_windows_removed':int((ambiguous & ~unknown).sum()),
        'common_windows_removed':int(common_exclusion.sum()),
        'zero_duration_rows':zero_rows,
        'identified_positive_window_species_pairs_removed':int(labels[common_exclusion].sum()),
        'known_positive_windows_removed_any_species':int((labels[common_exclusion].sum(axis=1)>0).sum()),
        'discarded_tail_samples':sum(r['audio']['discarded_tail_samples'] for r in records),
        'tail_only_annotations':dict(tail_only),'annotation_rows':rows_total,
        'selected_recording_annotation_rows':rows_selected,'duplicate_annotation_rows':duplicate_rows,
        'N':N,'B':B,'K':K,'J':len(targets),'histories':20*len(targets),'workflow_rows':160*len(targets),
        'recordings':records,'eligibility':eligibility,'eligible_codes':eligible_codes,
        'species_permutation_indices':permutation,'selected_targets':task_table,
        'ordered_pool_id_sha256':canonical_hash(pool_ids.tolist()),
        'source_sha256':{str(p.relative_to(root)):sha(p) for p in [root/'data/source/annotations.csv',root/'data/source/species.csv',root/'data/source/recording_location.txt',root/'data/constructed/RECORDING_PERMUTATION.json',root/'data/constructed/RECORDING_SELECTION.json',root/'data/constructed/ANNOTATION_PREFLIGHT.json']}}
    destination=root/'data/constructed'
    recovery=destination/'AUDIO_RECOVERY_PROVENANCE.json'
    if recovery.exists():result['source_sha256'][str(recovery.relative_to(root))]=sha(recovery)
    if (destination/'CONSTRUCTION_MANIFEST.json').exists():
        previous=load(destination/'CONSTRUCTION_MANIFEST.json')
        # Byte-for-byte repeatability of arrays is checked separately; timestamps are not scientific inputs.
        previous.pop('utc',None); compare=dict(result);compare.pop('utc',None)
        if previous != compare:
            raise RuntimeError('Construction differs on resume')
        for name,digest in load(destination/'CONSTRUCTION_ARTIFACT_HASHES.json')['files'].items():
            if Path(name).name!=name or sha(destination/name)!=digest:raise RuntimeError('Construction artifact changed on resume')
        if not targets:raise RuntimeError('CONSTRUCTION_FAILURE_NO_ELIGIBLE_TARGETS: see construction report')
        return load(destination/'CONSTRUCTION_MANIFEST.json')
    selected_matrix=labels[keep][:,[code_index[c] for c in targets]]
    savez(destination/'pool.npz',window_ids=pool_ids,recording_index=window_record[keep],
          window_index=np.asarray(window_j,dtype=np.int64)[keep],
          target_codes=np.asarray(targets,dtype='U20'),labels=selected_matrix,
          recording_filenames=np.asarray([r['filename'] for r in records]))
    savez(destination/'label_construction.npz',raw_ids=np.asarray(window_ids),
          unknown_exclusion=unknown,zero_duration_exclusion=ambiguous,common_exclusion=common_exclusion,all_identified_codes=np.asarray(codes),all_identified_labels=labels)
    dump(destination/'CONSTRUCTION_MANIFEST.json',result)
    dump(destination/'CONSTRUCTION_ARTIFACT_HASHES.json',{
        'recipe_freeze_sha256':sha(root/'study.json'),
        'files':{name:sha(destination/name) for name in ['pool.npz','label_construction.npz','CONSTRUCTION_MANIFEST.json']}})
    report=['# Sapsucker Woods construction result','',
            f"Status: {result['status']}. No classifier outcomes were used in construction.",'',
            f"33 recordings; {N_raw:,} complete windows; {result['unknown_windows_removed']:,} unknown-overlap windows removed; {result['zero_duration_additional_windows_removed']:,} additional ambiguous-timestamp windows removed.",
            f"The source table contains {len(zero_rows)} zero-duration rows; {result['zero_duration_selected_annotation_rows']} belong to selected recordings. These mark {result['zero_duration_timestamp_windows']} distinct complete windows, including {result['zero_duration_windows_already_unknown']} already excluded by unknown-call overlap. The combined mask removes {result['common_windows_removed']:,} windows.",
            f"Common pool N={N:,}, unique-label cap B={B:,}, flagged size K={K:,}. Selected targets J={len(targets)}.",'',
            '| Target | eBird code | Positive windows | Recordings with positives |',
            '|---|---|---:|---:|']
    report += [f"| {t['common_name']} | {t['code']} | {t['positive_windows']} | {t['positive_recordings']} |" for t in task_table]
    report += ['',f"{result['identified_positive_window_species_pairs_removed']:,} identified-positive window/species pairs were removed by the common exclusion mask.",
               'All eligibility counts and random permutations are retained. Counts are benchmark-construction information, not learner observations.','']
    (root/'CONSTRUCTION_REPORT.md').write_text('\n'.join(report))
    if not targets:raise RuntimeError('CONSTRUCTION_FAILURE_NO_ELIGIBLE_TARGETS: see construction report')
    return result

if __name__ == '__main__':
    parser=argparse.ArgumentParser();parser.add_argument('--root',type=Path,default=ROOT);parser.add_argument('--preflight',action='store_true');args=parser.parse_args()
    try:
        if args.preflight:
            result=preflight(args.root.resolve());print('ANNOTATION_PREFLIGHT_PASS',result['annotation_rows'],'rows;',len(result['zero_duration_rows']),'zero-duration rows to exclude by window')
        else:
            result=construct(args.root.resolve())
            print(json.dumps({k:result[k] for k in ['status','N','B','K','J','histories','workflow_rows','zero_duration_additional_windows_removed']},indent=2))
    except Exception:
        dump(args.root/'diagnostics/construction_failure.json',{'utc':utc(),'traceback':traceback.format_exc(),'automatic_redraw':False})
        raise


## 3. Fit LR/SVM and acquire labels
Only purchased labels enter fitting. LR uses raw embeddings; SVM standardization is fitted on the paid subset. Acquisition follows LR entropy with deterministic ties.

In [ ]:
%%writefile learning.py
"""Raw-LR acquisition; only purchased labels enter fitting or selection."""
from support import np, core, array_hash
import math, random, time, warnings
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

def training_arrays(ids,observed):
    idx=np.asarray([i for i,w in enumerate(ids) if w in observed],dtype=np.int64)
    y=np.asarray([observed[ids[i]] for i in idx],dtype=np.int64)
    if set(y.tolist())!={0,1}:raise ValueError('Two classes required')
    return idx,y

def native_predictions(decision):
    a=np.asarray(decision,dtype=np.float64)
    if not np.isfinite(a).all():raise ValueError('Nonfinite decision')
    return (a>0).astype(np.int8)

def fit_lr(X,ids,observed,seed):
    started=time.monotonic();idx,y=training_arrays(ids,observed)
    model=LogisticRegression(C=1,solver='lbfgs',penalty='l2',class_weight='balanced',fit_intercept=True,
                             tol=1e-4,max_iter=200,random_state=seed)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always');model.fit(X[idx],y)
    if model.classes_.tolist()!=[0,1]:raise RuntimeError('Invalid LR classes')
    p=np.asarray(model.predict_proba(X)[:,1],dtype=np.float64)
    decision=np.asarray(model.decision_function(X),dtype=np.float64)
    pred=native_predictions(decision)
    assert np.array_equal(pred,model.predict(X))
    if not np.isfinite(p).all():raise RuntimeError('Nonfinite LR probability')
    meta={'model':'raw_lr','count':len(idx),'positive_training':int(y.sum()),'fit_order':'pool',
          'training_indices_sha256':array_hash(idx),'training_labels_sha256':array_hash(y),
          'score_sha256':array_hash(p),'n_iter':model.n_iter_.tolist(),
          'warnings':[str(w.message) for w in caught],'seconds':time.monotonic()-started}
    return {'probability':p,'decision':decision,'prediction':pred,'coef':model.coef_.copy(),
            'intercept':model.intercept_.copy()},meta

def fit_svm(X,ids,observed,seed):
    started=time.monotonic();idx,y=training_arrays(ids,observed)
    scaler=StandardScaler().fit(X[idx])
    model=SVC(C=1,kernel='rbf',gamma='scale',class_weight='balanced',probability=False,tol=1e-3,
              shrinking=True,max_iter=-1,cache_size=200,random_state=seed)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always');model.fit(scaler.transform(X[idx]),y)
    if model.classes_.tolist()!=[0,1] or model.fit_status_!=0:raise RuntimeError('Invalid SVM fit')
    scores=np.asarray(model.decision_function(scaler.transform(X)),dtype=np.float64)
    if not np.isfinite(scores).all():raise RuntimeError('Nonfinite SVM scores')
    arrays={'scores':scores,'scaler_mean':scaler.mean_,'scaler_var':scaler.var_,'scaler_scale':scaler.scale_,
            'support_vectors':model.support_vectors_,'support':model.support_,'dual_coef':model.dual_coef_,
            'intercept':model.intercept_}
    meta={'model':'standardized_rbf_svm','count':len(idx),'positive_training':int(y.sum()),'fit_order':'pool',
          'training_indices_sha256':array_hash(idx),'training_labels_sha256':array_hash(y),
          'score_sha256':array_hash(scores),'classes':model.classes_.tolist(),'gamma':float(model._gamma),
          'n_support':model.n_support_.tolist(),'n_iter':model.n_iter_.tolist(),'scaler_fit_count':int(scaler.n_samples_seen_),
          'warnings':[str(w.message) for w in caught],'seconds':time.monotonic()-started,
          'array_sha256':{k:array_hash(v) for k,v in arrays.items()}}
    return arrays,meta

def entropy_query(scores,ids,observed,k):
    def entropy(i):
        p=min(1-1e-12,max(1e-12,float(scores[i])))
        return -p*math.log(p)-(1-p)*math.log(1-p)
    unseen=[i for i,w in enumerate(ids) if w not in observed]
    return sorted(unseen,key=lambda i:(-entropy(i),ids[i]))[:k]

def acquire(ids,X,oracle,seed,horizon=3000,batch=500,checkpoint=None,resume=None):
    """A completed snapshot is the only resumable acquisition boundary."""
    fits=[];snapshots=[];queries=[]
    if resume:
        order=list(map(int,resume['order']));startup=int(resume['startup'])
        snapshots=resume['snapshots'];fits=resume['fits'];queries=resume['queries']
        for j,i in enumerate(order):oracle.purchase(ids[i],'paid_search_startup' if j<startup else 'training')
        if resume.get('status')=='STARTUP_UNAVAILABLE':
            return {'order':order,'startup':startup,'snapshots':[],'fits':[],'queries':[], 'status':'STARTUP_UNAVAILABLE'}
    else:
        shuffled=list(range(len(ids)));random.Random(seed).shuffle(shuffled)
        order=[];classes=set()
        for i in shuffled[:horizon]:
            classes.add(oracle.purchase(ids[i],'paid_search_startup'));order.append(i)
            if len(classes)==2:break
        startup=len(order)
        if len(classes)<2:
            return {'order':order,'startup':startup,'snapshots':[],'fits':[],'queries':[], 'status':'STARTUP_UNAVAILABLE'}
    while True:
        if not snapshots or int(snapshots[-1]['count'])!=len(order):
            arrays,meta=fit_lr(X,ids,oracle.observed,seed)
            arrays['count']=len(order);snapshots.append(arrays);fits.append(meta)
            state={'order':order.copy(),'startup':startup,'snapshots':snapshots,'fits':fits,'queries':queries,'status':'CANDIDATE'}
            if checkpoint:checkpoint(state)
            print('LR_FIT',seed,len(order),round(meta['seconds'],2),flush=True)
        if len(order)==horizon:break
        selected=entropy_query(snapshots[-1]['probability'],ids,oracle.observed,min(batch,horizon-len(order)))
        if not selected or len(set(selected))!=len(selected):raise RuntimeError('Invalid acquisition batch')
        queries.append({'before':len(order),'indices':selected,'score_sha256':array_hash(snapshots[-1]['probability'])})
        for i in selected:
            if ids[i] in oracle.observed:raise RuntimeError('Duplicate acquisition')
            oracle.purchase(ids[i],'training');order.append(i)
    return {'order':order,'startup':startup,'snapshots':snapshots,'fits':fits,'queries':queries,'status':'CANDIDATE'}


## 4. Define stopping rules
Stabilizing predictions uses three adjacent kappa comparisons; confidence stopping requires three strict decreases. Detection freezes the current snapshot.

In [ ]:
%%writefile stopping.py
"""Permitted-information stopping statistics; no labels or model-fitting API."""
from fractions import Fraction
import math


def binary_kappa(first, second):
    a, b = tuple(first), tuple(second)
    if not a or len(a) != len(b):
        raise ValueError('nonempty equal-length prediction vectors required')
    if any(x not in (0, 1) for x in a + b):
        raise ValueError('binary predictions required')
    n = len(a)
    agreement = sum(x == y for x, y in zip(a, b))
    na, nb = sum(a), sum(b)
    expected_numerator = na * nb + (n - na) * (n - nb)
    denominator = n * n - expected_numerator
    if denominator == 0:
        return None
    return Fraction(n * agreement - expected_numerator, denominator)


def sp_first_trigger(prediction_vectors):
    vectors = [tuple(v) for v in prediction_vectors]
    if not vectors:
        return None, []
    # Validate even a lone first vector, for which no pair is available.
    binary_kappa(vectors[0], vectors[0])
    kappas = []
    first = None
    for j in range(1, len(vectors)):
        kappas.append(binary_kappa(vectors[j - 1], vectors[j]))
        latest = kappas[-3:]
        if (first is None and len(latest) == 3
                and all(k is not None for k in latest)
                and sum(latest, Fraction(0)) / 3 >= Fraction(99, 100)):
            first = j
    return first, kappas


def binary_confidence(probabilities):
    values = tuple(float(p) for p in probabilities)
    if not values or any(not math.isfinite(p) or not 0 <= p <= 1 for p in values):
        raise ValueError('finite probabilities in [0,1] required')
    entropy_sum = 0.0
    # Ordered float64 scalar operations are part of the declared convention.
    for p in values:
        if p not in (0.0, 1.0):
            entropy_sum += -p * math.log2(p) - (1.0 - p) * math.log2(1.0 - p)
    return 1.0 - entropy_sum / len(values)


def confidence_first_trigger(confidences):
    values = [float(c) for c in confidences]
    if any(not math.isfinite(c) or not 0 <= c <= 1 for c in values):
        raise ValueError('finite confidence values in [0,1] required')
    decreases = 0
    for j in range(1, len(values)):
        decreases = decreases + 1 if values[j] < values[j - 1] else 0
        if decreases == 3:
            return j
    return None


def select_endpoint(snapshot_counts, trigger_index=None, *, fixed=False, horizon=3000):
    counts = tuple(snapshot_counts)
    if (not counts or any(type(c) is not int or c <= 0 for c in counts)
            or any(b <= a for a, b in zip(counts, counts[1:]))
            or counts[-1] != horizon):
        raise ValueError('increasing paid counts ending at the horizon required')
    if fixed:
        if trigger_index is not None:
            raise ValueError('fixed endpoint does not take a trigger')
        return len(counts) - 1, 'fixed_horizon'
    if trigger_index is None:
        return len(counts) - 1, 'horizon_without_trigger'
    if type(trigger_index) is not int or not 3 <= trigger_index < len(counts):
        raise ValueError('three adjacent comparisons are required')
    status = 'triggered_at_horizon' if counts[trigger_index] == horizon else 'triggered_before_horizon'
    return trigger_index, status


## 5. Choose endpoints and evaluate one final audit
Candidate ranking and audit design precede retrospective outcomes. The exact audit mathematics is in helpers/audit.py.

In [ ]:
%%writefile evaluate.py
"""Candidate/design selection precedes evaluator-only metrics."""
from support import *
from dataclasses import asdict
from stopping import sp_first_trigger,binary_confidence,confidence_first_trigger,select_endpoint
METHODS={'lr':'LR-RAW-FA','svm':'SVM-STD-FA','equal_blend':'LR50-SVM50-FA','3to1_blend':'LR75-SVM25-FA'}

def monitoring_indices(N,seed,size=2000):
    rng=np.random.Generator(np.random.PCG64(np.random.SeedSequence([991220000,seed])))
    return np.sort(rng.choice(N,size=min(N,size),replace=False))

def endpoints(state,monitor,horizon=3000):
    snapshots=state['snapshots'];counts=[int(s['count']) for s in snapshots]
    if not snapshots:return {},{'status':'STARTUP_UNAVAILABLE'}
    sp,kappas=sp_first_trigger([s['prediction'][monitor].tolist() for s in snapshots])
    confidence=[binary_confidence(s['probability'][monitor]) for s in snapshots]
    cd=confidence_first_trigger(confidence)
    chosen={'fixed':select_endpoint(counts,fixed=True,horizon=horizon),
            'sp':select_endpoint(counts,sp,horizon=horizon),
            'confidence_decline':select_endpoint(counts,cd,horizon=horizon)}
    detail={'counts':counts,'confidence':confidence,'adjacent_kappas':[str(k) if k is not None else None for k in kappas],
            'first_sp_index':sp,'first_confidence_index':cd,'selected':{k:{'index':i,'status':s,'count':counts[i]} for k,(i,s) in chosen.items()}}
    return chosen,detail

def evaluate_candidate(ids,truth,order,startup,lr,svm,workflow,seed,stop_status,cap=5889,flagged_count=2355):
    # The only object constructed from outcomes before design is a paid oracle.
    candidate=core.freeze_candidate(ids,lr,svm,METHODS[workflow['ranker']],flagged_count)
    purchased=[ids[int(i)] for i in order];oracle=core.PaidOracle(truth,cap)
    for j,w in enumerate(purchased):oracle.purchase(w,'paid_search_startup' if j<startup else 'training')
    U,L,F=set(ids),set(purchased),set(candidate.flagged_ids)
    audit_order=core.common_order(ids,seed,991230000)
    trace=core.OneAttempt().run(candidate,purchased,oracle,audit_order)
    trace.update(seed=seed,workflow=workflow['id'])
    pre=len(L|F);D=U-(L|F)
    row={'seed':seed,'workflow':workflow['id'],'ranker':workflow['ranker'],'stop':workflow['stop'],
         'stop_status':stop_status,'training_count':len(order),'startup_paid':startup,'candidate_available':True,
         'flagged_count':len(F),'overlap':len(L&F),'pre_attention':pre,'flagged_id_hash':candidate.flagged_id_hash,
         'purchased_history_hash':core.canonical_hash([[w,truth[w]] for w in purchased]),
         'realized_total_attention':trace['actual_total_attention'],'realized_accepted':trace['terminal_state']=='ACCEPT',
         'terminal_reason':trace['terminal_reason'],'N':len(ids),'attention_cap':cap}
    # All evaluator-only access follows the frozen candidate and audit attempt.
    M=int(sum(truth[w] for w in D));T=int(sum(truth[w] for w in F));b=int(sum(truth[w] for w in L-F))
    if pre>cap:
        total=T+b+M;adequate=bool(total>0 and 10*T>=9*total)
        row.update(design=None,design_status='CENSUS_UNAFFORDABLE',boundary_check=None,
                   evaluator_only_M=M,evaluator_only_T=T,evaluator_only_b=b,
                   evaluator_only_recall=T/total if total else 0.,evaluator_only_adequate=adequate,
                   conditional_accept_probability=0.,correct_accept_probability=0.,false_accept_probability=0.,
                   expected_audit_inspected=0.,expected_total_attention=len(order),planned_total_attention=len(order))
    else:
        design=core.plan_audit(d=int(trace['unseen_d']),T=int(trace['T']),b_known=int(trace['b_known']),pre_attention=pre,cap=cap)
        row.update(design=asdict(design),design_status=design.status,boundary_check=core.verify_design(design))
        row.update(core.retrospective_metrics(design,evaluator_only_M=M))
        row.update(evaluator_only_T=T,evaluator_only_b=b)
    row['unaccepted_adequacy']=float(row['evaluator_only_adequate'])*(1-row['conditional_accept_probability'])
    row['realized_correct_accept']=row['realized_accepted'] and row['evaluator_only_adequate']
    row['realized_false_accept']=row['realized_accepted'] and not row['evaluator_only_adequate']
    assert row['planned_total_attention']<=cap and row['realized_total_attention']<=cap
    return row,trace,candidate.flagged_ids

def unavailable_row(ids,order,startup,workflow,seed,cap=5889):
    row={'seed':seed,'workflow':workflow['id'],'ranker':workflow['ranker'],'stop':workflow['stop'],
         'stop_status':'STARTUP_UNAVAILABLE','training_count':len(order),'startup_paid':startup,'candidate_available':False,
         'design':None,'design_status':'STARTUP_UNAVAILABLE','boundary_check':None,
         'evaluator_only_recall':None,'evaluator_only_adequate':None,'conditional_accept_probability':0.,
         'correct_accept_probability':0.,'false_accept_probability':0.,'unaccepted_adequacy':0.,
         'expected_total_attention':len(order),'planned_total_attention':len(order),'realized_total_attention':len(order),
         'realized_accepted':False,'realized_correct_accept':False,'realized_false_accept':False,'N':len(ids),'attention_cap':cap}
    trace={'seed':seed,'workflow':workflow['id'],'training_charged_ids':[ids[i] for i in order],
           'census_charged_ids':[],'audit_charged_ids':[],'audit_observed_labels':[],'observed_hits':0,
           'actual_total_attention':len(order),'audit_inspected':0,'terminal_state':'STOP_UNACCEPTED','terminal_reason':'STARTUP_UNAVAILABLE',
           'ledger':[{'window_id':ids[i],'purpose':'paid_search_startup','charge':1} for i in order]}
    return row,trace


## 6. Generate and cache Perch embeddings
The original waveform normalization and batches are explicit here. Each completed recording is verified and saved to the persistent cache.

In [ ]:
%%writefile embedding.py
"""Perch inference on original waveforms, with persistent recording checkpoints."""
import os, sys
os.environ['OMP_NUM_THREADS']='1'
os.environ['TF_NUM_INTEROP_THREADS']='1'
os.environ['TF_NUM_INTRAOP_THREADS']='1'
sys.modules['pandas']=None
sys.modules['pyarrow']=None
from pathlib import Path
import argparse, platform, subprocess, tarfile, tempfile, shutil, time
import numpy as np
import soundfile as sf
from artifacts import ROOT, load, dump, sha, savez
import cache

def normalize(waveforms):
    a=np.asarray(waveforms,dtype=np.float32).copy()
    a-=np.mean(a,axis=-1,keepdims=True,dtype=np.float32)
    peak=np.max(np.abs(a),axis=-1,keepdims=True)
    return np.divide(a,peak,out=np.zeros_like(a),where=peak>0)*np.float32(.25)

def load_model(device,tf):
    spec=load(ROOT/'models.json')[device];directory=ROOT/'data/models'/device
    if not (directory/'saved_model.pb').exists():
        directory.parent.mkdir(parents=True,exist_ok=True)
        archive=directory.with_suffix('.tar.gz');temporary=archive.with_suffix('.download')
        subprocess.run(['curl','-sS','--max-time','900','--fail','--location',spec['url'],'-o',str(temporary)],check=True)
        os.replace(temporary,archive)
        with tarfile.open(archive) as t:
            for member in t.getmembers():
                if not (member.isfile() or member.isdir()) or member.name.startswith('/') or '..' in Path(member.name).parts:raise RuntimeError('Unsafe model archive')
            t.extractall(directory)
    actual={str(p.relative_to(directory)):sha(p) for p in directory.rglob('*') if p.is_file()}
    if actual!=spec['model_files']:raise RuntimeError('Pinned model file checksum mismatch')
    return tf.saved_model.load(str(directory)).signatures['serving_default'],spec

def main():
    parser=argparse.ArgumentParser();parser.add_argument('--device',choices=['cpu','gpu'],required=True)
    parser.add_argument('--cache',type=Path,required=True);parser.add_argument('--prepare',action='store_true');args=parser.parse_args()
    import tensorflow as tf
    gpus=tf.config.list_physical_devices('GPU')
    if args.device=='gpu':
        if not gpus:raise RuntimeError('Select a GPU Colab runtime for GPU embedding generation')
        if not tf.sysconfig.get_build_info().get('is_cuda_build'):raise RuntimeError('GPU generation requires a CUDA TensorFlow build')
        tf.config.set_visible_devices(gpus[:1],'GPU');tf.config.experimental.set_memory_growth(gpus[0],True)
    else:tf.config.set_visible_devices([],'GPU')
    tf.config.threading.set_intra_op_parallelism_threads(1);tf.config.threading.set_inter_op_parallelism_threads(1)
    runtime={'python':platform.python_version(),'numpy':np.__version__,'tensorflow':tf.__version__,
             'soundfile':sf.__version__,'libsndfile':sf.__libsndfile_version__,'device':args.device,
             'threads':1,'batch_size':32}
    if args.device=='gpu':
        runtime['gpu_model']=tf.config.experimental.get_device_details(gpus[0]).get('device_name','GPU')
        runtime['cuda']=tf.sysconfig.get_build_info().get('cuda_version')
        runtime['cudnn']=tf.sysconfig.get_build_info().get('cudnn_version')
        driver=subprocess.run(['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader'],check=True,capture_output=True,text=True)
        runtime['driver_version']=driver.stdout.splitlines()[0].strip()
    expected={'python':'3.11.15','numpy':'2.4.6','tensorflow':'2.20.0','soundfile':'0.14.0'}
    if any(runtime[k]!=v for k,v in expected.items()):raise RuntimeError('Embedding dependency versions differ')
    infer,spec=load_model(args.device,tf)
    manifest={'device':args.device,'runtime':runtime,'model_id':spec['id'],'model_files':spec['model_files'],
              'source_sha256':sha(ROOT/'embedding.py')}
    path=ROOT/'EMBEDDING_EXECUTION_MANIFEST.json'
    if path.exists() and load(path)!=manifest:raise RuntimeError('Partial generation runtime changed; use explicit regeneration')
    if not path.exists():
        for n in [1,32]:
            values=np.asarray(infer(inputs=tf.zeros((n,160000),dtype=tf.float32))['embedding'])
            if values.shape!=(n,1536) or not np.isfinite(values).all():raise RuntimeError('Synthetic inference failed')
        dump(path,manifest)
    if args.prepare:print('EMBEDDING_RUNTIME_READY');return
    c=load(ROOT/'data/constructed/CONSTRUCTION_MANIFEST.json')
    with np.load(ROOT/'data/constructed/pool.npz',allow_pickle=False) as z:
        ids=z['window_ids'];ri=z['recording_index'];js=z['window_index']
    destination=ROOT/'data/embeddings';destination.mkdir(parents=True,exist_ok=True)
    provenance={'embedding_manifest_sha256':sha(path),'pool_sha256':sha(ROOT/'data/constructed/pool.npz'),'runtime':runtime}
    cache.save(ROOT,args.cache);all_embeddings=[]
    for i,record in enumerate(c['recordings']):
        name=record['filename'];selected=np.flatnonzero(ri==i)
        output=destination/(name+'.npz');receipt=output.with_suffix('.json')
        if receipt.exists():
            r=load(receipt)
            if r['provenance']!=provenance or sha(output)!=r['sha256']:raise RuntimeError('Cached recording differs')
            with np.load(output,allow_pickle=False) as z:
                if not np.array_equal(z['window_ids'],ids[selected]):raise RuntimeError('Cached ID order differs')
                vectors=z['embeddings']
        else:
            audio=ROOT/'data/audio'/name
            if sha(audio)!=record['audio']['sha256']:raise RuntimeError('Original audio checksum differs')
            batches=[]
            with sf.SoundFile(audio) as f:
                for start in range(0,len(selected),32):
                    waveforms=[]
                    for j in js[selected[start:start+32]]:
                        f.seek(int(j)*160000);wave=f.read(160000,dtype='float32',always_2d=True)
                        if wave.shape!=(160000,record['audio']['channels']) or not np.isfinite(wave).all():raise RuntimeError('Invalid decoded waveform')
                        waveforms.append(wave.mean(axis=1,dtype=np.float32))
                    waveforms=normalize(np.stack(waveforms));values=np.asarray(infer(inputs=waveforms)['embedding'])
                    if values.ndim>2:values=values.mean(axis=tuple(range(1,values.ndim-1)))
                    values=values.reshape(len(waveforms),-1).astype(np.float32)
                    if values.shape!=(len(waveforms),1536) or not np.isfinite(values).all():raise RuntimeError('Invalid embedding batch')
                    batches.append(values)
            vectors=np.concatenate(batches) if batches else np.empty((0,1536),np.float32)
            savez(output,window_ids=ids[selected],embeddings=vectors)
            dump(receipt,{'provenance':provenance,'sha256':sha(output),'audio_sha256':record['audio']['sha256']})
            cache.save(ROOT,args.cache);print('RECORDING_EMBEDDED',i+1,name,flush=True)
        all_embeddings.append(vectors)
    savez(destination/'SSW_EMBEDDINGS.npz',window_ids=ids,embeddings=np.concatenate(all_embeddings))
    dump(destination/'EMBEDDING_RECEIPT.json',{'provenance':provenance,'sha256':sha(destination/'SSW_EMBEDDINGS.npz')})
    cache.verify(ROOT,True,True);cache.save(ROOT,args.cache);print('EMBEDDINGS_COMPLETE')
if __name__=='__main__':main()


## 7. Execute every declared history
Each paid trajectory is shared across its eight workflows. Full IDs, scores and ledgers are retained; completed histories resume without refitting.

In [ ]:
%%writefile run_experiment.py
"""Frozen, resumable external target-history execution. No outcome selection."""
from support import *
from learning import acquire,fit_svm
from evaluate import endpoints,monitoring_indices,evaluate_candidate,unavailable_row
import argparse, time, traceback

def source_hashes():
    project=json.loads((ROOT/'project.json').read_text())
    names=set(project['scientific_source_sha256'])|set(project['helper_source_sha256'])
    files=[ROOT/n for n in sorted(names)]+[ROOT/'STUDY_MANIFEST.json',ROOT/'project.json']
    return {str(p.relative_to(ROOT)):sha(p) for p in files}

def check_execution():
    from artifacts import check_recipe
    check_recipe()
    if sha(ROOT/'EXECUTION_MANIFEST.json')!=(ROOT/'EXECUTION_MANIFEST.json.sha256').read_text().split()[0]:raise RuntimeError('Execution manifest sidecar mismatch')
    m=json.loads((ROOT/'EXECUTION_MANIFEST.json').read_text())
    for name,digest in m['source_sha256'].items():
        if sha(ROOT/name)!=digest:raise RuntimeError('Execution source changed: '+name)
    validate_runtime();return m

def save_state(folder,state,execution_hash):
    # Store immutable snapshots once, then atomically advance the checkpoint index.
    for j,(a,meta) in enumerate(zip(state['snapshots'],state['fits'])):
        path=folder/f'lr_{j:02d}.npz'
        if not path.exists():savez(path,**a)
    receipt={k:state[k] for k in ['order','startup','fits','queries','status']}
    receipt['snapshot_files']=[{'path':f'lr_{j:02d}.npz','sha256':sha(folder/f'lr_{j:02d}.npz')} for j in range(len(state['snapshots']))]
    receipt['execution_sha256']=execution_hash;dump(folder/'checkpoint.json',receipt)

def read_state(folder,execution_hash):
    p=folder/'checkpoint.json'
    if not p.exists():return None
    state=json.loads(p.read_text());assert state['execution_sha256']==execution_hash
    snapshots=[]
    for item in state.pop('snapshot_files'):
        assert sha(folder/item['path'])==item['sha256'];snapshots.append(loadz(folder/item['path']))
    state['snapshots']=snapshots;return state

def run_seed(ds,seed,target):
    execution=check_execution();execution_hash=sha(ROOT/'EXECUTION_MANIFEST.json')
    folder=ROOT/'runs'/target/str(seed);folder.mkdir(parents=True,exist_ok=True)
    marker=folder/'COMPLETE.json'
    if marker.exists():
        complete=json.loads(marker.read_text());assert complete['execution_sha256']==execution_hash
        for name,digest in complete['files'].items():assert sha(folder/name)==digest
        print('REUSED_COMPLETE',seed,flush=True);return
    started=time.monotonic();ids=ds.window_ids
    # Monitoring is selected before any query or fit; its labels are never read.
    monitor=monitoring_indices(len(ids),seed)
    study=json.loads((ROOT/'STUDY_MANIFEST.json').read_text())
    cap=study['constants']['unique_label_cap'];K=study['constants']['flagged_count']
    truth=dict(zip(ids,ds.labels.tolist()));oracle=core.PaidOracle(truth,cap)
    state=acquire(ids,ds.embeddings,oracle,seed,checkpoint=lambda s:save_state(folder,s,execution_hash),resume=read_state(folder,execution_hash))
    save_state(folder,state,execution_hash)
    chosen,detail=endpoints(state,monitor)
    dump(folder/'endpoint_selection.json',detail)
    # Fixed first-trigger decisions exist on disk before evaluator-only outcomes.
    workflows=json.loads((ROOT/'STUDY_MANIFEST.json').read_text())['workflows']
    svm_by={};svm_metadata={}
    needed=sorted({chosen[w['stop']][0] for w in workflows if w['ranker']!='lr'}) if chosen else []
    for index in needed:
        count=int(state['snapshots'][index]['count']);p=folder/f'svm_{count}.npz';receipt=p.with_suffix('.json')
        if receipt.exists():
            meta=json.loads(receipt.read_text());assert meta['execution_sha256']==execution_hash and sha(p)==meta['cache_sha256']
            arrays=loadz(p)
        else:
            observed={ids[i]:oracle.observed[ids[i]] for i in state['order'][:count]}
            arrays,meta=fit_svm(ds.embeddings,ids,observed,seed)
            savez(p,**arrays);meta.update(cache_sha256=sha(p),execution_sha256=execution_hash)
            dump(receipt,meta);print('SVM_FIT',seed,count,round(meta['seconds'],2),flush=True)
        svm_by[index]=arrays['scores'];svm_metadata[str(count)]=meta
    rows=[];traces=[];flagged={}
    for w in workflows:
        if not chosen:
            row,trace=unavailable_row(ids,state['order'],state['startup'],w,seed,cap=cap)
        else:
            index,status=chosen[w['stop']];snap=state['snapshots'][index];count=int(snap['count'])
            lr=snap['probability'];svm=svm_by.get(index,lr)
            row,trace,flags=evaluate_candidate(ids,truth,state['order'][:count],state['startup'],lr,svm,w,seed,status,cap=cap,flagged_count=K)
            flagged[w['id']]=np.asarray(flags)
        row['target']=target;trace['target']=target
        rows.append(row);traces.append(trace)
    savez(folder/'identities.npz',pool_ids=np.asarray(ids),purchased_indices=np.asarray(state['order'],dtype=np.int64),
          purchased_ids=np.asarray([ids[i] for i in state['order']]),purchased_labels=np.asarray([oracle.observed[ids[i]] for i in state['order']],dtype=np.int8),
          monitor_indices=monitor,**{'flagged_'+k:v for k,v in flagged.items()})
    dump(folder/'rows.json',rows);dump(folder/'traces.json',traces)
    dump(folder/'run_receipt.json',{'seed':seed,'status':'EXECUTED_PENDING_INDEPENDENT_VERIFICATION','seconds':time.monotonic()-started,
         'lr_fits':state['fits'],'svm_fits':svm_metadata,'runtime':runtime(),'execution_sha256':execution_hash,
         'new_model_fit_count':len(state['fits'])+len(needed),'warnings':[v for m in state['fits']+list(svm_metadata.values()) for v in m['warnings']]})
    names=['checkpoint.json','endpoint_selection.json','identities.npz','rows.json','traces.json','run_receipt.json']
    names += [f'lr_{j:02d}.npz' for j in range(len(state['snapshots']))]
    names += [f'svm_{count}'+suffix for count in svm_metadata for suffix in ['.npz','.json']]
    files={name:sha(folder/name) for name in names}
    dump(marker,{'seed':seed,'status':'EXECUTION_COMPLETE_VERIFICATION_PENDING','execution_sha256':execution_hash,'files':files})
    print('SEED_EXECUTED',seed,'rows',len(rows),'seconds',round(time.monotonic()-started,1),flush=True)

def main():
    parser=argparse.ArgumentParser();parser.add_argument('--target');parser.add_argument('--seed',type=int);args=parser.parse_args()
    check_execution();study=json.loads((ROOT/'STUDY_MANIFEST.json').read_text())
    if args.target and args.target not in [t['code'] for t in study['targets']]:raise ValueError('Unfrozen target')
    assigned=[(t['code'],seed) for t in study['targets'] for seed in t['seeds']]
    if args.seed and not any(s==args.seed and (args.target is None or t==args.target) for t,s in assigned):raise ValueError('Unfrozen seed')
    for target in study['targets']:
        code=target['code']
        if args.target and args.target!=code:continue
        ds,receipt=load_inputs(code);dump(ROOT/'results'/f'input_receipt_{code}.json',receipt)
        for seed in target['seeds']:
            if args.seed and args.seed!=seed:continue
            try:run_seed(ds,seed,code)
            except Exception:
                dump(ROOT/'diagnostics'/f'{code}_{seed}_integrity_failure.json',{'target':code,'seed':seed,'error_type':sys.exc_info()[0].__name__,'outcome_excluded':False})
                raise
    print('REQUESTED_FROZEN_HISTORIES_EXECUTED_VERIFY_NEXT',flush=True)
if __name__=='__main__':main()


## 8. Calculate paired results
The primary contrast is equal-rank blend minus LR. The bootstrap samples histories independently within each fixed target, preserving workflow pairing.

In [ ]:
%%writefile analyze.py
"""Frozen target-stratified paired analysis, including every history."""
from artifacts import ROOT, load, dump, sha, savez
from collections import Counter
import csv, hashlib
import numpy as np

NAMES={'fixed_lr':'Fixed LR','fixed_svm':'Fixed SVM','fixed_equal_blend':'Fixed equal blend',
       'fixed_3to1_blend':'Fixed 3:1 blend','stability_lr':'Stabilizing predictions: LR',
       'stability_blend':'Stabilizing predictions: equal blend','confidence_lr':'Confidence decline: LR',
       'confidence_blend':'Confidence decline: equal blend'}
CONTRASTS=[('fixed_equal_blend','fixed_lr'),('fixed_svm','fixed_lr'),('fixed_3to1_blend','fixed_lr'),
           ('fixed_equal_blend','fixed_3to1_blend'),('stability_lr','fixed_lr'),('confidence_lr','fixed_lr'),
           ('stability_blend','fixed_equal_blend'),('confidence_blend','fixed_equal_blend'),
           ('stability_blend','stability_lr'),('confidence_blend','confidence_lr')]

def bootstrap_indices(J,resamples=20000):
    rng=np.random.Generator(np.random.PCG64(991240000))
    return np.stack([rng.integers(0,20,size=(resamples,20)) for _ in range(J)])

def paired_estimate(delta,cost,indices):
    # delta and cost are [target, replicate]; target identity is fixed.
    dy=np.stack([delta[t][indices[t]].mean(axis=1) for t in range(len(delta))])
    dc=np.stack([cost[t][indices[t]].mean(axis=1) for t in range(len(cost))])
    def record(a,b,boot_a,boot_b):
        return {'mean_delta_Y':float(np.mean(a)),'mean_delta_cost':float(np.mean(b)),
                'Y_percentile_95':np.quantile(boot_a,[.025,.975],method='linear').tolist(),
                'cost_percentile_95':np.quantile(boot_b,[.025,.975],method='linear').tolist(),
                'wins':int(np.sum(a>1e-10)),'ties':int(np.sum(np.abs(a)<=1e-10)),
                'losses':int(np.sum(a< -1e-10))}
    per=[record(delta[t],cost[t],dy[t],dc[t]) for t in range(len(delta))]
    aggregate=record(delta,cost,dy.mean(axis=0),dc.mean(axis=0))
    return per,aggregate

def summarize(group,target,workflow):
    available=[r for r in group if r['candidate_available']]
    def mean(key,rows=group):
        return float(np.mean([r[key] for r in rows])) if rows else None
    result={'target':target,'workflow':workflow,'name':NAMES[workflow],'histories':len(group),
            'candidates_available':len(available),'unavailable':len(group)-len(available),
            'adequate':sum(bool(r['evaluator_only_adequate']) for r in available),
            'inadequate':sum(not r['evaluator_only_adequate'] for r in available),
            'mean_recall_available':mean('evaluator_only_recall',available),
            'mean_expected_attention':mean('expected_total_attention'),
            'mean_realized_attention':mean('realized_total_attention'),
            'mean_training_count':mean('training_count'),
            'max_planned_attention':max(r['planned_total_attention'] for r in group),
            'max_realized_attention':max(r['realized_total_attention'] for r in group),
            'design_statuses':dict(Counter(r['design_status'] for r in group)),
            'stop_statuses':dict(Counter(r['stop_status'] for r in group)),
            'cutoffs':dict(Counter(str(r['design']['c']) for r in group if r['design_status']=='AUDIT')),
            'mean_T_available':mean('evaluator_only_T',available),
            'mean_M_available':mean('evaluator_only_M',available),
            'mean_overlap_available':mean('overlap',available)}
    for key in ['correct_accept_probability','false_accept_probability','conditional_accept_probability',
                'unaccepted_adequacy','realized_accepted','realized_correct_accept','realized_false_accept']:
        result['mean_'+key]=mean(key)
    result['adequacy_frequency']=result['adequate']/len(group)
    result['availability_frequency']=len(available)/len(group)
    return result

def main(root=ROOT):
    study=load(root/'STUDY_MANIFEST.json');out=root/'results'
    verification=load(out/'verification.json')
    if verification['status']!='PASS':raise RuntimeError('Independent verification must pass')
    rows=load(out/'history_results.json');targets=study['targets'];J=len(targets)
    by={(r['target'],r['seed'],r['workflow']):r for r in rows}
    if J==0 or len(rows)!=160*J or len(by)!=160*J:raise RuntimeError('Incomplete frozen row set')
    indices=bootstrap_indices(J)
    savez(out/'bootstrap_indices.npz',indices=indices,target_codes=np.asarray([t['code'] for t in targets]))
    summary=[]
    for target in targets:
        for w in study['workflows']:
            group=[by[(target['code'],seed,w['id'])] for seed in target['seeds']]
            summary.append(summarize(group,target['code'],w['id']))
    aggregates=[]
    for w in study['workflows']:
        group=[by[(t['code'],seed,w['id'])] for t in targets for seed in t['seeds']]
        row=summarize(group,'equal_target_weight',w['id'])
        # The probability and attention means equal the target-weighted means
        # because all 20 histories remain in every stratum. Recall is explicitly
        # conditional on availability, so average target-specific recall means.
        means=[s['mean_recall_available'] for s in summary if s['workflow']==w['id'] and s['mean_recall_available'] is not None]
        row['mean_recall_available']=float(np.mean(means)) if means else None
        row['targets_with_available_recall']=len(means);row['targets']=J
        aggregates.append(row)
    paired=[]
    for j,(left,right) in enumerate(CONTRASTS):
        delta=np.asarray([[by[(t['code'],s,left)]['correct_accept_probability']-by[(t['code'],s,right)]['correct_accept_probability'] for s in t['seeds']] for t in targets])
        cost=np.asarray([[by[(t['code'],s,left)]['expected_total_attention']-by[(t['code'],s,right)]['expected_total_attention'] for s in t['seeds']] for t in targets])
        per,aggregate=paired_estimate(delta,cost,indices)
        for i,value in enumerate(per):
            value.update(target=targets[i]['code'],delta_Y=delta[i].tolist(),delta_cost=cost[i].tolist())
        paired.append({'left':left,'right':right,'role':'primary' if j==0 else 'secondary_descriptive',
                       'by_target':per,'aggregate':aggregate})
    primary=paired[0]
    cap=study['constants']['unique_label_cap']
    gates={'mean_delta_Y_at_least_0_10':primary['aggregate']['mean_delta_Y']>=.10,
           'approximate_95_lower_positive':primary['aggregate']['Y_percentile_95'][0]>0,
           'integrity':True,'all_costs_within_cap':all(r['planned_total_attention']<=cap and r['realized_total_attention']<=cap for r in rows)}
    decision={'status':'EXTERNAL_COMPARISON_COMPLETE','targets':J,'histories':20*J,'workflow_rows':160*J,
              'primary_practical_criterion_met':all(gates.values()),'gates':gates,
              'primary_target_losses':[x['target'] for x in primary['by_target'] if x['mean_delta_Y']<0],
              'primary_target_losses_at_least_0_10':[x['target'] for x in primary['by_target'] if x['mean_delta_Y']<=-.10],
              'additional_seeds_or_dataset_run':False,'bootstrap_indices_sha256':hashlib.sha256(indices.tobytes()).hexdigest(),
              'execution_sha256':sha(root/'EXECUTION_MANIFEST.json'),
              'interpretation':'Acquisition randomization conditional on the fixed selected target tasks and external pool; approximate percentile interval.'}
    trigger_summary=[]
    for t in targets:
        for method in ['stability_lr','confidence_lr']:
            group=[by[(t['code'],s,method)] for s in t['seeds']]
            trigger_summary.append({'target':t['code'],'workflow':method,
                                    'statuses':dict(Counter(r['stop_status'] for r in group)),
                                    'endpoint_counts':[r['training_count'] for r in group],
                                    'first_trigger_count':sum(r['stop_status'].startswith('triggered_') for r in group)})
    mechanisms=[]
    for r in rows:
        keys=['target','seed','workflow','candidate_available','training_count','overlap','pre_attention',
              'evaluator_only_T','evaluator_only_b','evaluator_only_M','evaluator_only_recall',
              'evaluator_only_adequate','conditional_accept_probability','correct_accept_probability',
              'false_accept_probability','expected_audit_inspected','expected_total_attention','design_status']
        entry={k:r.get(k) for k in keys};entry['design']=r['design'];mechanisms.append(entry)
    for name,value in [('summary_by_target.json',summary),('summary_aggregate.json',aggregates),
                       ('paired_results.json',paired),('trigger_summary.json',trigger_summary),
                       ('mechanisms.json',mechanisms),('decision.json',decision)]:dump(out/name,value)
    flat=[]
    for r in rows:
        v={k:x for k,x in r.items() if not isinstance(x,(dict,list))}
        v.update({'design_'+k:x for k,x in (r['design'] or {}).items()});flat.append(v)
    with (out/'history_results.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=sorted(set().union(*(r.keys() for r in flat))))
        writer.writeheader();writer.writerows(flat)
    lines=['# Sapsucker Woods external comparison','',
           f"Completed {J} target tasks, {20*J} acquisition histories and {160*J} final workflow rows and audit traces. All declared histories remain in the analysis. Independent integrity checks passed.",'',
           '| Workflow | Adequate / histories | Correct acceptance | False acceptance | Expected windows |',
           '|---|---:|---:|---:|---:|']
    for s in aggregates:
        lines.append(f"| {s['name']} | {s['adequate']}/{s['histories']} | {100*s['mean_correct_accept_probability']:.2f}% | {100*s['mean_false_accept_probability']:.3f}% | {s['mean_expected_attention']:.2f} |")
    a=primary['aggregate']
    lines+=['',f"Primary equal-blend minus LR effect: **{100*a['mean_delta_Y']:.3f} percentage points**, approximate target-stratified paired percentile 95% interval [{100*a['Y_percentile_95'][0]:.3f}, {100*a['Y_percentile_95'][1]:.3f}]. Expected-attention difference: {a['mean_delta_cost']:.2f} windows.",'',
            f"The prespecified practical benchmark (at least 10 percentage points, positive interval lower endpoint and passing integrity/budget checks) is {'met' if all(gates.values()) else 'not met'}. The continuous effect and its uncertainty remain the primary finding.",'',
            '| Target | LR correct acceptance | Equal blend correct acceptance | Paired effect (pp) | Approximate 95% interval (pp) |',
            '|---|---:|---:|---:|---:|']
    for t,effect in zip(targets,primary['by_target']):
        lookup={s['workflow']:s for s in summary if s['target']==t['code']}
        lines.append(f"| {t['common_name']} ({t['code']}) | {100*lookup['fixed_lr']['mean_correct_accept_probability']:.2f}% | {100*lookup['fixed_equal_blend']['mean_correct_accept_probability']:.2f}% | {100*effect['mean_delta_Y']:.3f} | [{100*effect['Y_percentile_95'][0]:.3f}, {100*effect['Y_percentile_95'][1]:.3f}] |")
    lines+=['',f"Targets with a negative primary mean effect: {', '.join(decision['primary_target_losses']) or 'none'}. Targets with losses of at least 10 percentage points: {', '.join(decision['primary_target_losses_at_least_0_10']) or 'none'}.",'',
            'Correct acceptance is a conditional probability multiplied by exact candidate adequacy, averaged over every history. It is not the observed fraction of single sampled audit traces that passed. Non-acceptance is not proof of classifier inadequacy.','',
            'Uncertainty reflects acquisition randomization for the selected target tasks on the fixed external pool. It does not cover new recordings, species or environments. The approximate bootstrap interval and the 5% per-workflow audit guarantee are different quantities; no family-wide zero-false-acceptance claim is made.','',
            f"Independent checks covered all {verification['traces']} traces, exact risk boundaries, adjacent audit allocations, paid-label accounting, acquisition choices, rankings and stopping behavior. Maximum probability discrepancy {verification['max_probability_absolute_error']:.3g}; maximum expected-cost discrepancy {verification['max_expected_cost_absolute_error']:.3g} windows. These saved-output checks do not independently retrain the classifiers.",'',
            'All secondary contrasts, target-level adequacy/false-acceptance/attention summaries, mechanism counts, full score arrays, purchased IDs and trigger statuses are retained. Identical horizon fallback workflows remain separate outcomes.','']
    (root/'REPORT.md').write_text('\n'.join(lines))
    print('ANALYSIS_COMPLETE',decision,flush=True)

if __name__=='__main__':main()


## 9. Check cache and install only the required environments

In [ ]:
for name, expected in spec['scientific_source_sha256'].items():
    if hashlib.sha256((RUN_ROOT/name).read_bytes()).hexdigest()!=expected:
        raise RuntimeError('Scientific implementation changed: '+name)
def run(python, script, *arguments):
    env=dict(os.environ,OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1',VECLIB_MAXIMUM_THREADS='1',NUMEXPR_NUM_THREADS='1')
    if str(python)==globals().get('PYTHONS',{}).get('embedding'):
        env.update(json.loads((RUNTIMES/'embedding_environment.json').read_text()))
    p=subprocess.Popen([str(python),str(RUN_ROOT/script),*map(str,arguments)],cwd=RUN_ROOT,env=env,
                       stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    try:
        for line in p.stdout: print(line,end='',flush=True)
        if p.wait():raise RuntimeError(script+' failed; review its output before continuing')
    finally:
        if p.poll() is None:p.terminate();p.wait()
CACHE=Path(DRIVE_CACHE_PATH).expanduser();CACHE.mkdir(parents=True,exist_ok=True)
args=['restore','--directory',CACHE]+(['--regenerate'] if REGENERATE_EMBEDDINGS else [])
run(sys.executable,'cache.py',*args)
CACHED=json.loads((RUN_ROOT/'CACHE_STATUS.json').read_text())['status']=='complete'
AUDIO_CACHE=CACHE/'audio'
AUDIO_DIRECTORY=Path(AUDIO_BACKUP_DIRECTORY).expanduser() if AUDIO_BACKUP_DIRECTORY.strip() else AUDIO_CACHE
AUDIO_ARGS=['--directory',AUDIO_DIRECTORY]+(['--required'] if AUDIO_BACKUP_DIRECTORY.strip() else [])
AUDIO_FOUND=False
if not CACHED:
    run(sys.executable,'audio_store.py','restore',*AUDIO_ARGS,'--metadata-only')
    AUDIO_FOUND=json.loads((RUN_ROOT/'AUDIO_RESTORE_STATUS.json').read_text())['status']!='missing'
    if AUDIO_FOUND:run(sys.executable,'construction.py','--preflight')
RUNTIMES=RUN_ROOT.parent/'environments'
run(sys.executable,'environment.py','--directory',RUNTIMES,'--mode','analysis','--embedding-device',EMBEDDING_DEVICE)
PYTHONS=json.loads((RUNTIMES/'interpreters.json').read_text())
print('Using verified cached embeddings' if CACHED else 'Generating embeddings from original audio')


## 10. Restore or download audio, then inspect window construction
This stage validates all annotation rows before audio transfer, restores verified recordings when available, and reports the usable pool before Perch setup. A complete embedding cache skips these operations.

In [ ]:
if not CACHED:
    args=['--archive',ORIGINAL_ARCHIVE_PATH] if ORIGINAL_ARCHIVE_PATH.strip() else []
    if not AUDIO_FOUND:run(PYTHONS['classifier'],'dataset.py','--inventory-only',*args)
    run(sys.executable,'construction.py','--preflight')
    if AUDIO_FOUND:run(sys.executable,'audio_store.py','restore',*AUDIO_ARGS)
    run(PYTHONS['classifier'],'dataset.py','--audio-cache',AUDIO_CACHE,*args)
    run(PYTHONS['classifier'],'construction.py')
print((RUN_ROOT/'CONSTRUCTION_REPORT.md').read_text())
print('POOL_READY — inspect the report before continuing to embeddings.')


## 11. Generate or verify embeddings and freeze the inputs
Perch installation and model loading occur only after successful construction. Keep the same GPU runtime throughout if generating fresh embeddings.

In [ ]:
construction_result=json.loads((RUN_ROOT/'data/constructed/CONSTRUCTION_MANIFEST.json').read_text())
if construction_result['status']!='TASKS_CONSTRUCTED' or construction_result['recipe_freeze_sha256']!=spec['study_sha256']:
    raise RuntimeError('Complete section 10 successfully before starting Perch setup.')
if not CACHED:
    run(sys.executable,'environment.py','--directory',RUNTIMES,'--mode','full','--embedding-device',EMBEDDING_DEVICE)
    PYTHONS=json.loads((RUNTIMES/'interpreters.json').read_text())
    run(PYTHONS['embedding'],'embedding.py','--device',EMBEDDING_DEVICE,'--cache',CACHE,'--prepare')
    run(PYTHONS['embedding'],'embedding.py','--device',EMBEDDING_DEVICE,'--cache',CACHE)
run(PYTHONS['classifier'],'cache.py','verify')
run(PYTHONS['classifier'],'freeze.py')


## 12. Run the comparison and independent checks

In [ ]:
run(PYTHONS['classifier'],'run_experiment.py')
run(PYTHONS['classifier'],'verify.py')
run(PYTHONS['classifier'],'analyze.py')
run(PYTHONS['statistics'],'reproduce_statistics.py')
print((RUN_ROOT/'REPORT.md').read_text())
print(json.loads((RUN_ROOT/'STUDY_MANIFEST.json').read_text())['reproduction_scope'])


## 13. Export scientific results
The export contains code, input hashes, arrays and verified outcomes. It excludes runtime environments, working paths, diagnostic logs and notebook outputs.

In [ ]:
run(PYTHONS['classifier'],'export_results.py','--directory',CACHE/'result_exports')
receipt=json.loads((RUN_ROOT/'EXPORT_RECEIPT.json').read_text())
OUTPUT=CACHE/'result_exports'/receipt['filename']
print('Verified results saved:', OUTPUT.name)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(OUTPUT))
